In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import RandomOverSampler        
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import seaborn as sns


In [2]:
# 1. Load GloVe Embeddings
def load_glove_embeddings(glove_file_path):
    print(f"Loading GloVe embeddings from {glove_file_path}...")
    embeddings_index = {}
    try:
        with open(glove_file_path, encoding='utf-8') as f:
            for line in f:
                values = line.split()
                word = values[0]
                # Handle cases where a line might be malformed
                try:
                    coefs = np.asarray(values[1:], dtype='float32')
                    embeddings_index[word] = coefs
                except ValueError:
                    continue
        print(f"Found {len(embeddings_index)} word vectors.")
        return embeddings_index
    except FileNotFoundError:
        print(f"Error: File not found at {glove_file_path}")
        print("Please ensure 'glove.6B.100d.txt' is in the correct directory.")
        return {}

# 2. Generate Averaged Embeddings (Pure Python/Numpy)
def get_averaged_glove_features(texts, embeddings_index, embedding_dim=100):
    """
    Converts a list of text strings into a list of averaged GloVe vectors.
    """
    features = []
    for text in tqdm(texts, desc="Vectorizing texts"):
        if not isinstance(text, str):
            text = ""
            
        # Simple tokenization: lowercase and split by whitespace
        words = text.lower().split()
        
        # Filter for words present in GloVe
        valid_vectors = [embeddings_index[w] for w in words if w in embeddings_index]
        
        if valid_vectors:
            # Average the vectors
            avg_vector = np.mean(valid_vectors, axis=0)
        else:
            # If no words found (or empty text), return zero vector
            avg_vector = np.zeros(embedding_dim)
            
        features.append(avg_vector)
        
    return np.array(features)

In [3]:
# glove_path = 'dolma_300_2024_1.2M.100_combined.txt' 

# embeddings_index = load_glove_embeddings(glove_path)

# if embeddings_index:
#     # Load Data
#     print("Loading dataset...")
#     df = pd.read_csv("processed_datasets/WELFake_Dataset_processed.csv")
    
#     # Ensure text column is string
#     df['combined_text'] = df['combined_text'].astype(str)
    
#     texts = df['combined_text'].tolist()
#     labels = df['label'].values

#     # FIX: Determine embedding dimension from the loaded data automatically
#     # This prevents mismatch between loaded vectors and the zero-vector fallback
#     if len(embeddings_index) > 0:
#         sample_key = next(iter(embeddings_index))
#         embedding_dim = len(embeddings_index[sample_key])
#         print(f"Detected embedding dimension: {embedding_dim}")
#     else:
#         embedding_dim = 100 # Fallback default

#     # Generate Features
#     X_glove = get_averaged_glove_features(texts, embeddings_index, embedding_dim)

#     # Create DataFrame
#     glove_feature_cols = [f'glove_{i}' for i in range(embedding_dim)]
#     glove_df = pd.DataFrame(X_glove, columns=glove_feature_cols)
#     glove_df['label'] = labels

#     # Save
#     output_path = 'preprocessed_datasets/glove_features_averaged.csv'
#     glove_df.to_csv(output_path, index=False)
#     print(f"Averaged GloVe features saved to {output_path}")

In [4]:
def scale_data(dataframe, oversample=False):
  x = dataframe[dataframe.columns[:-1]].values
  y = dataframe[dataframe.columns[-1]].values

  scaler = StandardScaler()
  x = scaler.fit_transform(x)

  if oversample:
    ros = RandomOverSampler()
    x, y = ros.fit_resample(x, y)

  data = np.hstack((x, np.reshape(y, (-1, 1))))

  return data, x, y

In [5]:
def split_dataset(df):
    train = df.sample(frac=0.8, random_state=42)
    test = df.drop(train.index)
    print(f"Total rows in dataset: {len(df)}")
    print(f"Total rows in train set: {len(train)}")
    print(f"Total rows in test set: {len(test)}")

    print("\nClass distribution in full dataset:")
    print(df['label'].value_counts())

    print("\nClass distribution in train set:")
    print(train['label'].value_counts())

    print("\nClass distribution in test set:")
    print(test['label'].value_counts())
    
    train, xtrain, ytrain = scale_data(train)
    test, xtest, ytest = scale_data(test)
    
    print("\n KNN \n")

    return train, xtrain, ytrain, test, xtest, ytest

In [6]:
def train_ml_models(xtrain, ytrain, xtest, ytest, svc_flag=1):
    print("\n KNN \n")
    knn_model = KNeighborsClassifier(n_neighbors=5)
    knn_model.fit(xtrain, ytrain)

    ypred = knn_model.predict(xtest)
    print(classification_report(ytest, ypred))
    
    print("\n Gaussian NB \n")
    nb_model = GaussianNB()
    nb_model = nb_model.fit(xtrain, ytrain)

    ypred = nb_model.predict(xtest)

    print(classification_report(ytest, ypred))
    
    print("\n Logistica Regression \n")
    lgr_model = LogisticRegression(max_iter=1000)
    lgr_model = lgr_model.fit(xtrain, ytrain)

    ypred = lgr_model.predict(xtest)

    print(classification_report(ytest, ypred))
    
    if svc_flag == 1:
        print("\n Support Vector Classifier \n")
        svc_model = SVC()
        svc_model = svc_model.fit(xtrain, ytrain)

        ypred = svc_model.predict(xtest)
        print(classification_report(ytest, ypred))
    else: print("SVC Flag is not 1, skipping SVC")
    
    print("\n XGBoost \n")
    xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
    xgb_model = xgb_model.fit(xtrain, ytrain)

    ypred = xgb_model.predict(xtest)
    print(classification_report(ytest, ypred))
    
    print("\n Random Forest Classifier \n")
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_model.fit(xtrain, ytrain)

    ypred = rf_model.predict(xtest)
    print(classification_report(ytest, ypred))

In [7]:
class CNNModel(nn.Module):
    def __init__(self, embedding_dim, num_filters=100, filter_sizes=[3, 4, 5]):
        super(CNNModel, self).__init__()
        self.embedding_dim = embedding_dim
        
        # Reshape input to (batch_size, 1, embedding_dim) for conv1d
        self.conv_layers = nn.ModuleList([
            nn.Conv1d(in_channels=1, out_channels=num_filters, kernel_size=fs)
            for fs in filter_sizes
        ])
        
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)
        
        # Fully connected layers
        self.fc1 = nn.Linear(num_filters * len(filter_sizes), 64)
        self.fc2 = nn.Linear(64, 1)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        # x shape: (batch_size, embedding_dim)
        x = x.unsqueeze(1)  # (batch_size, 1, embedding_dim)
        
        # Apply convolutions
        conv_outputs = []
        for conv in self.conv_layers:
            conv_out = self.relu(conv(x))  # (batch_size, num_filters, length)
            pooled = torch.max(conv_out, dim=2)[0]  # Max pooling
            conv_outputs.append(pooled)
        
        # Concatenate all conv outputs
        x = torch.cat(conv_outputs, dim=1)  # (batch_size, num_filters * len(filter_sizes))
        
        x = self.dropout(x)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.sigmoid(self.fc2(x))
        
        return x

In [8]:
class LSTMModel(nn.Module):
    def __init__(self, embedding_dim, hidden_dim=128, num_layers=2):
        super(LSTMModel, self).__init__()
        self.embedding_dim = embedding_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        self.lstm = nn.LSTM(input_size=embedding_dim, hidden_size=hidden_dim, 
                           num_layers=num_layers, batch_first=True, dropout=0.5)
        self.dropout = nn.Dropout(0.5)
        self.fc1 = nn.Linear(hidden_dim, 64)
        self.fc2 = nn.Linear(64, 1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        # x shape: (batch_size, embedding_dim)
        x = x.unsqueeze(1)  # (batch_size, 1, embedding_dim) - add sequence dimension
        
        # LSTM forward pass
        lstm_out, (hidden, cell) = self.lstm(x)  # hidden: (num_layers, batch_size, hidden_dim)
        
        # Use last hidden state
        x = hidden[-1]  # (batch_size, hidden_dim)
        
        x = self.dropout(x)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.sigmoid(self.fc2(x))
        
        return x

In [9]:
class CNNLSTMModel(nn.Module):
    def __init__(self, embedding_dim, num_filters=64, filter_sizes=[3, 4, 5], 
                 hidden_dim=128, num_layers=1):
        super(CNNLSTMModel, self).__init__()
        self.embedding_dim = embedding_dim
        
        # CNN layers
        self.conv_layers = nn.ModuleList([
            nn.Conv1d(in_channels=1, out_channels=num_filters, kernel_size=fs)
            for fs in filter_sizes
        ])
        
        self.relu = nn.ReLU()
        
        # LSTM layers - input is concatenated conv outputs
        cnn_output_dim = num_filters * len(filter_sizes)
        self.lstm = nn.LSTM(input_size=cnn_output_dim, hidden_size=hidden_dim,
                           num_layers=num_layers, batch_first=True, dropout=0.5)
        
        self.dropout = nn.Dropout(0.5)
        
        # Fully connected layers
        self.fc1 = nn.Linear(hidden_dim, 64)
        self.fc2 = nn.Linear(64, 1)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        # x shape: (batch_size, embedding_dim)
        x = x.unsqueeze(1)  # (batch_size, 1, embedding_dim)
        
        # Apply CNN
        conv_outputs = []
        for conv in self.conv_layers:
            conv_out = self.relu(conv(x))  # (batch_size, num_filters, length)
            pooled = torch.max(conv_out, dim=2)[0]  # Max pooling
            conv_outputs.append(pooled)
        
        x = torch.cat(conv_outputs, dim=1)  # (batch_size, cnn_output_dim)
        x = x.unsqueeze(1)  # (batch_size, 1, cnn_output_dim) - add sequence dimension
        
        # Apply LSTM
        lstm_out, (hidden, cell) = self.lstm(x)
        x = hidden[-1]  # (batch_size, hidden_dim)
        
        x = self.dropout(x)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.sigmoid(self.fc2(x))
        
        return x

In [10]:
def train_deep_learning_model(model, train_loader, test_loader, epochs=10, learning_rate=0.001, device='cpu'):
    """
    Train a PyTorch model and evaluate on test set.
    
    Parameters:
    - model: PyTorch model
    - train_loader: Training DataLoader
    - test_loader: Test DataLoader
    - epochs: Number of training epochs
    - learning_rate: Learning rate
    - device: 'cpu' or 'cuda'
    """
    
    model = model.to(device)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    print(f"Training on device: {device}")
    print(f"Model: {model.__class__.__name__}")
    print("=" * 50)
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        
        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(device).float()
            batch_y = batch_y.to(device).float().unsqueeze(1)
            
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        # Evaluate on test set
        model.eval()
        test_loss = 0.0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for batch_x, batch_y in test_loader:
                batch_x = batch_x.to(device).float()
                batch_y = batch_y.to(device).float().unsqueeze(1)
                
                outputs = model(batch_x)
                loss = criterion(outputs, batch_y)
                test_loss += loss.item()
                
                # Calculate accuracy
                predicted = (outputs > 0.5).float()
                correct += (predicted == batch_y).sum().item()
                total += batch_y.size(0)
        
        avg_train_loss = train_loss / len(train_loader)
        avg_test_loss = test_loss / len(test_loader)
        accuracy = correct / total
        
        print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {avg_train_loss:.4f} | Test Loss: {avg_test_loss:.4f} | Accuracy: {accuracy:.4f}")
    
    print("=" * 50)
    print(f"{model.__class__.__name__} training completed!")
    
    return model

# ==================== Wrapper Function ====================
def train_all_deep_learning_models(xtrain, ytrain, xtest, ytest, epochs=10, batch_size=32):
    """
    Train CNN, LSTM, and CNN-LSTM models.
    
    Parameters:
    - xtrain, ytrain: Training data and labels
    - xtest, ytest: Test data and labels
    - epochs: Number of training epochs
    - batch_size: Batch size for DataLoader
    """
    
    # Determine device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Convert to PyTorch tensors
    xtrain_tensor = torch.from_numpy(xtrain).float()
    ytrain_tensor = torch.from_numpy(ytrain).long()
    xtest_tensor = torch.from_numpy(xtest).float()
    ytest_tensor = torch.from_numpy(ytest).long()
    
    # Create DataLoaders
    train_dataset = TensorDataset(xtrain_tensor, ytrain_tensor)
    test_dataset = TensorDataset(xtest_tensor, ytest_tensor)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    embedding_dim = xtrain.shape[1]
    
    # Train CNN
    print("\n" + "=" * 50)
    print("TRAINING CNN MODEL")
    print("=" * 50)
    cnn_model = CNNModel(embedding_dim)
    cnn_model = train_deep_learning_model(cnn_model, train_loader, test_loader, epochs=epochs, device=device)
    
    # Train LSTM
    print("\n" + "=" * 50)
    print("TRAINING LSTM MODEL")
    print("=" * 50)
    lstm_model = LSTMModel(embedding_dim)
    lstm_model = train_deep_learning_model(lstm_model, train_loader, test_loader, epochs=epochs, device=device)
    
    # Train CNN-LSTM
    print("\n" + "=" * 50)
    print("TRAINING CNN-LSTM HYBRID MODEL")
    print("=" * 50)
    cnn_lstm_model = CNNLSTMModel(embedding_dim)
    cnn_lstm_model = train_deep_learning_model(cnn_lstm_model, train_loader, test_loader, epochs=epochs, device=device)
    
    return cnn_model, lstm_model, cnn_lstm_model

In [11]:
def train_deep_learning_model(model, train_loader, val_loader, test_loader, epochs=10, learning_rate=0.001, device='cpu'):
    """
    Train a PyTorch model with validation set and evaluate on test set.
    
    Parameters:
    - model: PyTorch model
    - train_loader: Training DataLoader
    - val_loader: Validation DataLoader
    - test_loader: Test DataLoader (NOT USED during training, only for final evaluation)
    - epochs: Number of training epochs
    - learning_rate: Learning rate
    - device: 'cpu' or 'cuda'
    """
    
    model = model.to(device)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    print(f"Training on device: {device}")
    print(f"Model: {model.__class__.__name__}")
    print("=" * 50)
    
    # Storage for history
    train_losses = []
    val_losses = []
    val_accuracies = []
    
    for epoch in range(epochs):
        # Training
        model.train()
        train_loss = 0.0
        
        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(device).float()
            batch_y = batch_y.to(device).float().unsqueeze(1)
            
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        avg_train_loss = train_loss / len(train_loader)
        train_losses.append(avg_train_loss)
        
        # ==================== VALIDATION ====================
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x = batch_x.to(device).float()
                batch_y = batch_y.to(device).float().unsqueeze(1)
                
                outputs = model(batch_x)
                loss = criterion(outputs, batch_y)
                val_loss += loss.item()
                
                # Calculate accuracy
                predicted = (outputs > 0.5).float()
                val_correct += (predicted == batch_y).sum().item()
                val_total += batch_y.size(0)
        
        avg_val_loss = val_loss / len(val_loader)
        val_accuracy = val_correct / val_total
        val_losses.append(avg_val_loss)
        val_accuracies.append(val_accuracy)
        
        print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Accuracy: {val_accuracy:.4f}")
    
    print("=" * 50)
    print(f"{model.__class__.__name__} training completed!")
    
    # Test-set eval
    print("\n" + "=" * 50)
    print(f"FINAL EVALUATION ON TEST SET - {model.__class__.__name__}")
    print("=" * 50)
    
    model.eval()
    all_predictions = []
    all_labels = []
    all_probabilities = []
    
    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x = batch_x.to(device).float()
            batch_y = batch_y.to(device).float().unsqueeze(1)
            
            outputs = model(batch_x)
            predicted = (outputs > 0.5).float()
            
            all_predictions.extend(predicted.cpu().numpy().flatten())
            all_labels.extend(batch_y.cpu().numpy().flatten())
            all_probabilities.extend(outputs.cpu().numpy().flatten())
    
    # Calculate metrics
    all_predictions = np.array(all_predictions)
    all_labels = np.array(all_labels)
    all_probabilities = np.array(all_probabilities)
    
    test_accuracy = np.mean(all_predictions == all_labels)
    test_auc = roc_auc_score(all_labels, all_probabilities)
    
    print(f"\nTest Accuracy: {test_accuracy:.4f}")
    print(f"Test AUC Score: {test_auc:.4f}")
    print("\nClassification Report:")
    print(classification_report(all_labels, all_predictions, target_names=['Fake (0)', 'Real (1)']))
    
    # Visualisation
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Loss curves
    axes[0, 0].plot(train_losses, label='Training Loss', marker='o')
    axes[0, 0].plot(val_losses, label='Validation Loss', marker='s')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title(f'{model.__class__.__name__} - Loss over Epochs')
    axes[0, 0].legend()
    axes[0, 0].grid(True)
    
    # Accuracy curve
    axes[0, 1].plot(val_accuracies, label='Validation Accuracy', marker='o', color='green')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Accuracy')
    axes[0, 1].set_title(f'{model.__class__.__name__} - Accuracy over Epochs')
    axes[0, 1].legend()
    axes[0, 1].grid(True)
    
    # Confusion Matrix
    cm = confusion_matrix(all_labels, all_predictions)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1, 0], cbar=False)
    axes[1, 0].set_xlabel('Predicted')
    axes[1, 0].set_ylabel('Actual')
    axes[1, 0].set_title(f'{model.__class__.__name__} - Confusion Matrix (Test Set)')
    axes[1, 0].set_xticklabels(['Fake (0)', 'Real (1)'])
    axes[1, 0].set_yticklabels(['Fake (0)', 'Real (1)'])
    
    # ROC Curve
    fpr, tpr, _ = roc_curve(all_labels, all_probabilities)
    axes[1, 1].plot(fpr, tpr, label=f'ROC Curve (AUC = {test_auc:.4f})', color='darkorange', lw=2)
    axes[1, 1].plot([0, 1], [0, 1], 'k--', lw=2, label='Random Classifier')
    axes[1, 1].set_xlabel('False Positive Rate')
    axes[1, 1].set_ylabel('True Positive Rate')
    axes[1, 1].set_title(f'{model.__class__.__name__} - ROC Curve (Test Set)')
    axes[1, 1].legend()
    axes[1, 1].grid(True)
    
    plt.tight_layout()
    plt.show()
    
    return model, {
        'train_losses': train_losses,
        'val_losses': val_losses,
        'val_accuracies': val_accuracies,
        'test_accuracy': test_accuracy,
        'test_auc': test_auc,
        'predictions': all_predictions,
        'labels': all_labels,
        'probabilities': all_probabilities
    }

In [12]:
def train_all_deep_learning_models(xtrain, ytrain, xtest, ytest, epochs=10, batch_size=32, val_split=0.2):
    """
    Train CNN, LSTM, and CNN-LSTM models with proper train/validation/test split.
    
    Parameters:
    - xtrain, ytrain: Training data and labels
    - xtest, ytest: Test data and labels (NOT used during training)
    - epochs: Number of training epochs
    - batch_size: Batch size for DataLoader
    - val_split: Fraction of training data to use for validation (default: 0.2)
    
    Verification:
    - Train set is partitioned into training and validation
    - Test set is NOT used during training, only for final evaluation
    """
    
    print("=" * 70)
    print("DATA SPLIT VERIFICATION")
    print("=" * 70)
    print(f"Training set size: {xtrain.shape[0]}")
    print(f"Test set size: {xtest.shape[0]}")
    print(f"Validation split: {val_split * 100}%")
    print(f"Effective training size: {int(xtrain.shape[0] * (1 - val_split))}")
    print(f"Effective validation size: {int(xtrain.shape[0] * val_split)}")
    print("✓ Test set will NOT be used during training phase")
    print("✓ Test set will ONLY be used for final evaluation")
    print("=" * 70 + "\n")
    
    # Determine device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Split training data into train and validation
    from sklearn.model_selection import train_test_split as sklearn_train_test_split
    xtrain_split, xval, ytrain_split, yval = sklearn_train_test_split(
        xtrain, ytrain, test_size=val_split, random_state=42, stratify=ytrain
    )
    
    # Convert to PyTorch tensors
    xtrain_tensor = torch.from_numpy(xtrain_split).float()
    ytrain_tensor = torch.from_numpy(ytrain_split).long()
    xval_tensor = torch.from_numpy(xval).float()
    yval_tensor = torch.from_numpy(yval).long()
    xtest_tensor = torch.from_numpy(xtest).float()
    ytest_tensor = torch.from_numpy(ytest).long()
    
    # Create DataLoaders
    train_dataset = TensorDataset(xtrain_tensor, ytrain_tensor)
    val_dataset = TensorDataset(xval_tensor, yval_tensor)
    test_dataset = TensorDataset(xtest_tensor, ytest_tensor)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    embedding_dim = xtrain.shape[1]
    
    models_history = {}
    
    # Train CNN
    print("\n" + "=" * 70)
    print("TRAINING CNN MODEL")
    print("=" * 70)
    cnn_model = CNNModel(embedding_dim)
    cnn_model, cnn_history = train_deep_learning_model(
        cnn_model, train_loader, val_loader, test_loader, 
        epochs=epochs, device=device
    )
    models_history['CNN'] = cnn_history
    
    # Train LSTM
    print("\n" + "=" * 70)
    print("TRAINING LSTM MODEL")
    print("=" * 70)
    lstm_model = LSTMModel(embedding_dim)
    lstm_model, lstm_history = train_deep_learning_model(
        lstm_model, train_loader, val_loader, test_loader, 
        epochs=epochs, device=device
    )
    models_history['LSTM'] = lstm_history
    
    # Train CNN-LSTM
    print("\n" + "=" * 70)
    print("TRAINING CNN-LSTM HYBRID MODEL")
    print("=" * 70)
    cnn_lstm_model = CNNLSTMModel(embedding_dim)
    cnn_lstm_model, cnn_lstm_history = train_deep_learning_model(
        cnn_lstm_model, train_loader, val_loader, test_loader, 
        epochs=epochs, device=device
    )
    models_history['CNN-LSTM'] = cnn_lstm_history
    
    return cnn_model, lstm_model, cnn_lstm_model, models_history

In [13]:
def run_complete_pipeline(dataset_csv_path, glove_file_path, output_dir='preprocessed_datasets', 
                          save_embeddings=1, train_ml=True, train_dl=True, 
                          dl_epochs=10, dl_batch_size=32, val_split=0.2):
    """
    Complete end-to-end pipeline: GloVe embeddings → ML models → Deep learning models
    
    Parameters:
    - dataset_csv_path: Path to processed dataset CSV
    - glove_file_path: Path to GloVe embeddings file
    - output_dir: Directory to save GloVe features
    - save_embeddings: If 1, saves GloVe features to CSV; if 0, skips saving
    - train_ml: If True, trains traditional ML models (KNN, NB, LR, SVC, XGBoost, RF)
    - train_dl: If True, trains deep learning models (CNN, LSTM, CNN-LSTM)
    - dl_epochs: Number of epochs for deep learning training
    - dl_batch_size: Batch size for deep learning DataLoaders
    - val_split: Validation split fraction (default: 0.2)
    
    Returns:
    - glove_df: GloVe feature DataFrame
    - xtrain, ytrain, xtest, ytest: Scaled train/test data
    - models_history: Dictionary containing histories from all deep learning models
    """
    
    print("\n" + "=" * 80)
    print("Starting complete GloVe + ML + DL pipeline")
    print("=" * 80 + "\n")
    
    # Step 1: Load GloVe embeddings
    print("Step 1: Loading GloVe embeddings")
    print("-" * 80)
    embeddings_index = load_glove_embeddings(glove_file_path)
    if not embeddings_index:
        print("Failed to load GloVe embeddings. Exiting pipeline.")
        return None
    
    # Step 2: Load dataset
    print("\nStep 2: Loading and processing dataset")
    print("-" * 80)
    df = pd.read_csv(dataset_csv_path)
    df['combined_text'] = df['combined_text'].astype(str)
    
    texts = df['combined_text'].tolist()
    labels = df['label'].values
    
    # Determine embedding dimension
    sample_key = next(iter(embeddings_index))
    embedding_dim = len(embeddings_index[sample_key])
    print(f"Detected embedding dimension: {embedding_dim}")
    
    # Step 3: Generate GloVe features
    print("\nStep 3: Generating averaged GloVe features")
    print("-" * 80)
    X_glove = get_averaged_glove_features(texts, embeddings_index, embedding_dim)
    
    # Create DataFrame
    glove_feature_cols = [f'glove_{i}' for i in range(embedding_dim)]
    glove_df = pd.DataFrame(X_glove, columns=glove_feature_cols)
    glove_df['label'] = labels
    
    # Conditional saving of embeddings
    if save_embeddings == 1:
        import os
        os.makedirs(output_dir, exist_ok=True)
        output_path = os.path.join(output_dir, 'glove_features_averaged.csv')
        glove_df.to_csv(output_path, index=False)
        print(f"Saved GloVe features to {output_path}")
    else:
        print("Skipping GloVe features save (save_embeddings=0)")
    
    # Step 4: Split dataset
    print("\nStep 4: Splitting dataset (80/20 train/test)")
    print("-" * 80)
    train, xtrain, ytrain, test, xtest, ytest = split_dataset(glove_df)
    
    # Step 5: Train ML models
    if train_ml:
        print("\nStep 5: Training traditional ML models")
        print("-" * 80)
        train_ml_models(xtrain, ytrain, xtest, ytest, 0)
    else:
        print("\nStep 5: Skipping traditional ML models (train_ml=False)")
    
    # Step 6: Train deep learning models
    models_history = {}
    if train_dl:
        print("\nStep 6: Training deep learning models with validation/test split")
        print("-" * 80)
        cnn_model, lstm_model, cnn_lstm_model, models_history = train_all_deep_learning_models(
            xtrain, ytrain, xtest, ytest, 
            epochs=dl_epochs, 
            batch_size=dl_batch_size,
            val_split=val_split
        )
    else:
        print("\nStep 6: Skipping deep learning models (train_dl=False)")
    
    # Pipeline complete
    print("\n" + "=" * 80)
    print("Pipeline completed successfully")
    print("=" * 80)
    print(f"GloVe embeddings generated: {embedding_dim} dimensions")
    print(f"Training samples: {len(xtrain)}")
    print(f"Test samples: {len(xtest)}")
    if train_ml:
        print("ML models trained (KNN, Naive Bayes, Logistic Regression, SVC, XGBoost, Random Forest)")
    if train_dl:
        print("DL models trained (CNN, LSTM, CNN-LSTM)")
        print(f"  - CNN test accuracy: {models_history['CNN']['test_accuracy']:.4f}")
        print(f"  - LSTM test accuracy: {models_history['LSTM']['test_accuracy']:.4f}")
        print(f"  - CNN-LSTM test accuracy: {models_history['CNN-LSTM']['test_accuracy']:.4f}")
    print("=" * 80 + "\n")
    
    return glove_df, xtrain, ytrain, xtest, ytest, models_history

In [ ]:
# Run full pipeline with defaults
glove_df, xtrain, ytrain, xtest, ytest, models_history = run_complete_pipeline(
    dataset_csv_path='processed_datasets/WELFake_Dataset_processed.csv',
    glove_file_path='dolma_300_2024_1.2M.100_combined.txt'
)


Starting complete GloVe + ML + DL pipeline

Step 1: Loading GloVe embeddings
--------------------------------------------------------------------------------
Loading GloVe embeddings from dolma_300_2024_1.2M.100_combined.txt...
Found 1200001 word vectors.

Step 2: Loading and processing dataset
--------------------------------------------------------------------------------
Detected embedding dimension: 300

Step 3: Generating averaged GloVe features
--------------------------------------------------------------------------------


Vectorizing texts: 100%|██████████| 72124/72124 [00:54<00:00, 1326.65it/s]


Saved GloVe features to preprocessed_datasets/glove_features_averaged.csv

Step 4: Splitting dataset (80/20 train/test)
--------------------------------------------------------------------------------
Total rows in dataset: 72124
Total rows in train set: 57699
Total rows in test set: 14425

Class distribution in full dataset:
label
1    37096
0    35028
Name: count, dtype: int64

Class distribution in train set:
label
1    29653
0    28046
Name: count, dtype: int64

Class distribution in test set:
label
1    7443
0    6982
Name: count, dtype: int64

 KNN 


Step 5: Training traditional ML models
--------------------------------------------------------------------------------

 KNN 

              precision    recall  f1-score   support

           0       0.81      0.91      0.86      6982
           1       0.90      0.80      0.85      7443

    accuracy                           0.85     14425
   macro avg       0.86      0.85      0.85     14425
weighted avg       0.86      0.85   

/Users/jordan/Desktop/Live_Projects/FYP_Documents/.venv/lib/python3.14/site-packages/xgboost/training.py:199: UserWarning: [23:15:57] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


              precision    recall  f1-score   support

           0       0.92      0.91      0.92      6982
           1       0.92      0.93      0.92      7443

    accuracy                           0.92     14425
   macro avg       0.92      0.92      0.92     14425
weighted avg       0.92      0.92      0.92     14425


 Random Forest Classifier 

              precision    recall  f1-score   support

           0       0.91      0.88      0.89      6982
           1       0.89      0.92      0.90      7443

    accuracy                           0.90     14425
   macro avg       0.90      0.90      0.90     14425
weighted avg       0.90      0.90      0.90     14425


Step 6: Training deep learning models with validation/test split
--------------------------------------------------------------------------------
DATA SPLIT VERIFICATION
Training set size: 57699
Test set size: 14425
Validation split: 20.0%
Effective training size: 46159
Effective validation size: 11539
✓ Test set w